02/09/2026

We now have a vector layer with possible candidates, that were identified through early models, but missed through manual labeling.

We need to join this with the previous version of labels, so we can create a new training dataset.

Save results directly to PostGIS.

In [11]:
import os
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Connect to PostGIS


Example of query:

`SELECT *` means all attributes, instead you can select which attributes to read

`als_ml_rezultati` is the selected **Schema**
`adaf_irish` is the desired **Table**

```SQL
'SELECT * FROM als_ml_rezultati.adaf_irish'
```

Another example:

```SQL
SELECT
    label_id,
    label_origin,
    annotation_round,
    source_model,
    geometry
FROM training_data.barrow_labels
WHERE label_origin = 'model_assisted'
```


In [3]:

connection_url = URL.create(
    drivername="postgresql",
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
    host="gisko.zrc-sazu.si",
    port=15432,
    database="STONE_delovno"
)

engine = create_engine(connection_url)

In [4]:
example_gdf = gpd.read_postgis(
    'SELECT * FROM als_ml_rezultati.possible_missed',
    engine
)

example_gdf.head()

,fid,geom,candidate_id,source_count,source_results,src_adaf_irish,src_adaf_retrained_1,src_adaf_retrained_2_256px,src_adaf_bih_0_256px,src_adaf_bih_0_128px,src_adaf_bih_0_512px,src_adaf_retrained_2_512px
0,1,"POLYGON ((254451.5 4752241.5, 254451.5 4752240...",14254,1,adaf_bih_0-256px,False,False,False,True,False,False,False
1,2,"POLYGON ((254351.5 4754725, 254351.5 4754724.5...",14257,1,adaf_bih_0-128px,False,False,False,False,True,False,False
2,3,"POLYGON ((254445 4754748.5, 254444 4754748.5, ...",14258,1,adaf_bih_0-128px,False,False,False,False,True,False,False
3,4,"POLYGON ((254472.5 4754759, 254472.5 4754758.5...",14259,1,adaf_bih_0-128px,False,False,False,False,True,False,False
4,5,"POLYGON ((254489 4754768, 254488.5 4754768, 25...",14260,1,adaf_bih_0-128px,False,False,False,False,True,False,False


In [7]:
len(example_gdf)

6414

# Load data

- possible_detections, but only those with source_count > 1


In [5]:
possible_labels_gdf = gpd.read_postgis(
    """
    SELECT *
    FROM als_ml_rezultati.possible_missed
    WHERE source_count > 1
    """,
    engine
)

In [6]:
len(possible_labels_gdf)

1016

- previous labels (to be updated)

In [9]:
old_gdf = gpd.read_postgis(
    """
    SELECT *
    FROM als_ml_podatki."gomile_2026-01-16"
    """,
    engine
)

In [10]:
len(old_gdf)

8616

To check for any shared polygon area between old_gdf and possible_labels_gdf, use an intersection overlay and test its area. This counts containment and duplicate polygons as overlaps but ignores polygons that only touch at an edge or point.

In [15]:
# Preserve the original feature indices
old_check = old_gdf[
    ["id", old_gdf.geometry.name]
].copy()

new_check = possible_labels_gdf[
    ["candidate_id", possible_labels_gdf.geometry.name]
].copy()

# Calculate intersections between the two layers
intersections = gpd.overlay(
    old_check,
    new_check,
    how="intersection",
    keep_geom_type=False
)

# Remove edge-only and point-only contacts
overlaps = intersections[
    intersections.geometry.area > 0
].copy()

print(f"Overlapping polygon pairs: {len(overlaps)}")
print(f"Total overlapping area: {overlaps.geometry.area.sum():.2f} m²")

Overlapping polygon pairs: 1
Total overlapping area: 143.81 m²


In [16]:
overlaps.head()

,id,candidate_id,geometry
0,5065,34820,"POLYGON ((266463.946 4756560, 266463.5 4756560..."


# Create new dataset

In [17]:
# IDs of old polygons that overlap new polygons
old_overlap_ids = overlaps["id"].dropna().unique()
old_overlap_ids

array([5065])

In [20]:
# Prepare old polygons, excluding those that overlap
old_part = old_gdf.loc[
    ~old_gdf["id"].isin(old_overlap_ids),
    ["id", old_gdf.geometry.name]
].copy()

old_part["candidate_id"] = pd.NA
old_part["source"] = "gomile_2026-01-16"
old_part["source_count"] = 0

old_part.head()

,id,geom,candidate_id,source,source_count
0,1,"MULTIPOLYGON (((286830.241 4752215.833, 286825...",<NA>,gomile_2026-01-16,0
1,2,"MULTIPOLYGON (((286803.549 4752395.348, 286805...",<NA>,gomile_2026-01-16,0
2,3,"MULTIPOLYGON (((286993.276 4752062.93, 286988....",<NA>,gomile_2026-01-16,0
3,4,"MULTIPOLYGON (((293763.246 4819367.658, 293763...",<NA>,gomile_2026-01-16,0
4,5,"MULTIPOLYGON (((273559.602 4734419.205, 273557...",<NA>,gomile_2026-01-16,0


In [22]:
# Prepare all new polygons
new_part = possible_labels_gdf[
    ["candidate_id", "source_count", possible_labels_gdf.geometry.name]
].copy()

new_part["id"] = pd.NA
new_part["source"] = "adaf_1"

new_part.head()

,candidate_id,source_count,geom,id,source
0,26886,7,"POLYGON ((261846 4760690, 261846 4760689, 2618...",<NA>,adaf_1
1,20323,3,"POLYGON ((257834 4791518.5, 257834 4791519, 25...",<NA>,adaf_1
2,20325,6,"POLYGON ((257672 4791542.5, 257672 4791543, 25...",<NA>,adaf_1
3,20917,7,"POLYGON ((257881.5 4791521.5, 257882 4791521.5...",<NA>,adaf_1
4,28227,7,"POLYGON ((262536.5 4765258, 262536.5 4765257.5...",<NA>,adaf_1


In [25]:
# Use the same column order
columns = [
    "id",
    "candidate_id",
    "source",
    "source_count",
    "geom",
]

old_part = old_part[columns]
new_part = new_part[columns]

# Merge the prepared layers
combined_gdf = gpd.GeoDataFrame(
    pd.concat([old_part, new_part], ignore_index=True),
    geometry="geom",
    crs=old_gdf.crs
)

print(f"Old overlapping polygons removed: {len(old_overlap_ids)}")
print(f"Old polygons retained: {len(old_part)}")
print(f"New polygons retained: {len(new_part)}")
print(f"Combined polygons: {len(combined_gdf)}")

combined_gdf.head()

Old overlapping polygons removed: 1
Old polygons retained: 8615
New polygons retained: 1016
Combined polygons: 9631


,id,candidate_id,source,source_count,geom
0,1,<NA>,gomile_2026-01-16,0,"MULTIPOLYGON (((286830.241 4752215.833, 286825..."
1,2,<NA>,gomile_2026-01-16,0,"MULTIPOLYGON (((286803.549 4752395.348, 286805..."
2,3,<NA>,gomile_2026-01-16,0,"MULTIPOLYGON (((286993.276 4752062.93, 286988...."
3,4,<NA>,gomile_2026-01-16,0,"MULTIPOLYGON (((293763.246 4819367.658, 293763..."
4,5,<NA>,gomile_2026-01-16,0,"MULTIPOLYGON (((273559.602 4734419.205, 273557..."


In [26]:
len(combined_gdf)

9631

# Save results

Save as GPKG file:

In [27]:
combined_gdf.to_file(
    r"r:\ML podatki\archaeology\gomile_2026-09-02.gpkg",
    driver="GPKG",
)

Save directly to PostGIS

In [28]:
combined_gdf.to_postgis(
    name="gomile_2026-09-02",
    con=engine,
    schema="als_ml_podatki",
    if_exists="replace",
    index=False,
)